In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
df = pd.read_csv("/content/data.csv")
df.head()

In [ ]:
df.info()

In [ ]:
df.drop("Unnamed: 32", axis=1, inplace=True)
df.drop("id", axis=1, inplace=True)

In [ ]:
sns.countplot(x='diagnosis', data=df)
plt.show()

In [ ]:
features = df.select_dtypes(include='number').columns

for col in features:
    sns.histplot(df[col], kde=True)
    plt.title(f'{col} Distribution')
    plt.xlabel(col)
    plt.ylabel('Frequency')
    plt.show()

In [ ]:
for col in features:
    plt.figure(figsize=(6,4))
    sns.boxplot(x='diagnosis', y=col, data=df)
    plt.title(f'{col} vs Diagnosis')
    plt.show()

In [ ]:
Q1 = df.drop("diagnosis", axis=1).quantile(0.25)
Q3 = df.drop("diagnosis", axis=1).quantile(0.75)
IQR = Q3 - Q1

outliers = ((df.drop("diagnosis", axis=1) < (Q1 - 1.5 * IQR)) |
            (df.drop("diagnosis", axis=1) > (Q3 + 1.5 * IQR)))

print("Number of outliers in each feature:")
print(outliers.sum())

In [ ]:
plt.figure(figsize=(6,5))
sns.scatterplot(x='radius_mean', y='area_mean', hue='diagnosis', data=df)
plt.title("Radius Mean vs Area Mean by Diagnosis")
plt.show()

In [ ]:
df['diagnosis'] = df['diagnosis'].map({'M':1, 'B':0})

In [ ]:
plt.figure(figsize=(18,14))

corr = df.corr(numeric_only=True)

sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    linewidths=0.5,
    annot_kws={"size":7}
)

plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)

plt.title("Correlation Heatmap", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
X = df.drop("diagnosis", axis=1)
y = df['diagnosis']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
def evaluate_model(model, model_name):

    y_pred = model.predict(X_test)

    print("\n" + "="*50)
    print(model_name)
    print("="*50)

    acc = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {acc:.4f}")

    print("\nClassification Report:\n")
    print(classification_report(y_test, y_pred))

    cm = confusion_matrix(y_test, y_pred)

    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f"{model_name} - Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()


In [ ]:
log_model = LogisticRegression(penalty=None, max_iter=1000)
log_model.fit(X_train, y_train)

evaluate_model(log_model, "Logistic Regression (No Regularization)")

In [ ]:
coef_log = pd.Series(log_model.coef_[0], index=X.columns)

coef_log.sort_values().plot(kind='barh', figsize=(8,6))
plt.title("Feature Importance - Logistic (No Regularization)")
plt.xlabel("Coefficient Value")
plt.ylabel("Features")
plt.show()

In [ ]:
# ---------------- Lasso Logistic Regression (L1) ----------------
lasso_model = LogisticRegression(
    penalty='l1',
    solver='liblinear',
    C=1.0,
    max_iter=1000
)

lasso_model.fit(X_train, y_train)

# Evaluation
evaluate_model(lasso_model, "Lasso Logistic Regression (L1)")

# ---------------- Feature Importance & Sparsity ----------------
import pandas as pd

lasso_coef = pd.Series(lasso_model.coef_[0], index=X.columns)

print("Number of zero coefficients in Lasso:", sum(lasso_coef == 0))

plt.figure(figsize=(8,6))
lasso_coef.sort_values().plot(kind='barh')
plt.title("Lasso Feature Importance")
plt.xlabel("Coefficient Value")
plt.ylabel("Features")
plt.show()

In [ ]:
ridge_model = LogisticRegression(penalty='l2', C=1.0, max_iter=1000)
ridge_model.fit(X_train, y_train)

evaluate_model(ridge_model, "Ridge Logistic Regression (L2)")

In [ ]:
coef_ridge = pd.Series(ridge_model.coef_[0], index=X.columns)

coef_ridge.sort_values().plot(kind='barh', figsize=(8,6))
plt.title("Feature Importance - Ridge Logistic Regression (L2)")
plt.xlabel("Coefficient Value")
plt.ylabel("Features")
plt.show()

In [ ]:
elastic_model = LogisticRegression(
    penalty='elasticnet',
    solver='saga',
    l1_ratio=0.5,
    C=1.0,
    max_iter=5000
)

elastic_model.fit(X_train, y_train)

evaluate_model(elastic_model, "Elastic Net Logistic Regression")

In [ ]:
coef_elastic = pd.Series(elastic_model.coef_[0], index=X.columns)

coef_elastic.sort_values().plot(kind='barh', figsize=(8,6))
plt.title("Feature Importance - Elastic Net Logistic Regression")
plt.xlabel("Coefficient Value")
plt.ylabel("Features")
plt.show()

In [ ]:
X_train_gd = np.c_[np.ones((X_train.shape[0], 1)), X_train]
X_test_gd  = np.c_[np.ones((X_test.shape[0], 1)), X_test]

# Sigmoid
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# Gradient Descent
def gradient_descent(X, y, alpha=0.01, iterations=5000):
    m, n = X.shape
    theta = np.zeros(n)

    for _ in range(iterations):
        h = sigmoid(X @ theta)
        gradient = (1/m) * (X.T @ (h - y))
        theta -= alpha * gradient

    return theta

# Prediction
def predict_gd(X, theta):
    return (sigmoid(X @ theta) >= 0.5).astype(int)


In [ ]:
theta = gradient_descent(X_train_gd, y_train.values)

In [ ]:
y_pred_gd = predict_gd(X_test_gd, theta)

print("\n" + "="*50)
print("Logistic Regression (Gradient Descent From Scratch)")
print("="*50)

print("Accuracy:", accuracy_score(y_test, y_pred_gd))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_gd))

cm = confusion_matrix(y_test, y_pred_gd)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title("Logistic Regression (GD) - Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
coef_gd = pd.Series(theta[1:], index=X.columns)

coef_gd.sort_values().plot(kind='barh', figsize=(8,6))
plt.title("Feature Importance - Logistic (Gradient Descent)")
plt.xlabel("Coefficient Value")
plt.ylabel("Features")
plt.show()

In [ ]:
y_pred_log = log_model.predict(X_test)
y_pred_ridge = ridge_model.predict(X_test)
y_pred_lasso = lasso_model.predict(X_test)
y_pred_elastic = elastic_model.predict(X_test)

In [ ]:
results = pd.DataFrame(columns=["Model", "Accuracy", "Precision", "Recall", "F1"])

models_predictions = {
    "Logistic": y_pred_log,
    "Gradient Descent": y_pred_gd,
    "Ridge": y_pred_ridge,
    "Lasso": y_pred_lasso,
    "Elastic Net": y_pred_elastic
}

for name, y_pred in models_predictions.items():
    results.loc[len(results)] = [
        name,
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred),
        recall_score(y_test, y_pred),
        f1_score(y_test, y_pred)
    ]

results